# The evasion gap

**Question:** does an off-the-shelf toxicity classifier survive trivial obfuscation?

**Threat model.** An adversary wants abusive text to stay readable to humans while scoring below the moderation threshold. No model access, no gradients, no ML — a keyboard and a Unicode table.

**The catch this notebook is built around.** The answer depends entirely on where the threshold sits. Pinning it to 95% recall drives it to the floor and makes the metric blind; pinning it to a 1% false-positive budget — what a platform actually ships — shows a 61% relative recall loss from a nine-line attack. Both are computed here, side by side.

The same run is reproducible headless via `python scripts/run_experiment.py`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import yaml

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from evasion_gap.attacks import ATTACKS, HOMOGLYPHS, ZWSP
from evasion_gap.data import load_corpus
from evasion_gap.metrics import rate_above
from evasion_gap.model import Scorer
from evasion_gap.pipeline import build_operating_point, run_attack_sweep
from evasion_gap.plots import plot_robustness

config = yaml.safe_load((ROOT / "config.yaml").read_text())
config

## 1. Data

Streamed from `civil_comments`: toxic = human-rated toxicity ≥ 0.8, benign ≤ 0.1. The benign set is not decoration — without it there is no false-positive budget, and without that there is no defensible threshold.

This cell is the slow one (~5 min): toxicity ≥ 0.8 is rare, so the stream scans a lot of rows.

In [ ]:
toxic, benign = load_corpus(**config["dataset"])
print(f"{len(toxic)} toxic / {len(benign)} benign")
toxic[0][:200]

In [ ]:
scorer = Scorer(
    model_id=config["model_id"],
    batch_size=config["eval"]["batch_size"],
    max_length=config["eval"]["max_length"],
)

toxic_scores = scorer(toxic)
benign_scores = scorer(benign)

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.hist(benign_scores, bins=40, alpha=0.6, label="benign")
ax.hist(toxic_scores, bins=40, alpha=0.6, label="toxic")
ax.set_xlabel("P(toxic)")
ax.set_ylabel("count")
ax.set_title("Score separation on clean text")
ax.legend()
plt.tight_layout()

**Read the left tail.** A meaningful share of genuinely toxic comments score near zero. That tail is what makes a recall-pinned threshold collapse in the next cell — and it is why the first version of this experiment reached the wrong conclusion.

## 2. Two operating points

- `high_recall` — threshold pinned to catch 95% of clean toxic text. Intuitive, and the trap.
- `fpr_1pct` — threshold pinned to a 1% false-positive budget. What a platform ships, because over-blocking benign users is the expensive error.

In [ ]:
operating_points = [
    build_operating_point(spec, toxic_scores, benign_scores)
    for spec in config["eval"]["operating_points"]
]

import pandas as pd
pd.DataFrame([op.as_dict() for op in operating_points])

The recall-pinned threshold lands near **0.02** — effectively the floor — and carries a **7% FPR**, which no platform would run. A rate measured there cannot move, whatever the attack does. That is a broken instrument, not a robust model.

### What the attacks look like

Every variant must stay readable. A transform that mangles text past human comprehension is not an evasion — it is noise, and it does not belong in the suite.

In [ ]:
sample = toxic[0][:80]
for name, fn in ATTACKS.items():
    print(f"{name:>12}: {fn(sample)}")

## 3. Attack sweep

Each variant is scored once and evaluated at both thresholds. The thresholds do not move between attacks — an operating point chosen on clean data is what actually ships.

In [ ]:
sweep = run_attack_sweep(scorer, toxic, operating_points, seed=config["seed"])
sweep

In [ ]:
fig, axes = plot_robustness(sweep, operating_points, outfile=ROOT / "results" / "robustness.png")

## 4. Failure inspection

The sweep says *how much* broke. The tokenizer says *why*.

In [ ]:
shipping = [op for op in operating_points if op.name == "fpr_1pct"][0]
worst = (
    sweep[sweep["operating_point"] == shipping.name]
    .sort_values("recall")
    .iloc[0]["attack"]
)
transform = ATTACKS[worst]
print(f"worst attack at {shipping.name}: {worst}\n")

for text in toxic[:3]:
    variant = transform(text)
    before, after = scorer([text])[0], scorer([variant])[0]
    print(f"{before:.3f} -> {after:.3f}")
    print(f"  tokens: {scorer.tokenizer.tokenize(variant)[:20]}\n")

### Why `zero_width` does nothing

Its scores match clean to three decimals — too exact to be robustness. The hypothesis: BERT's text cleaning strips category-`Cf` codepoints before tokenization, so the attack never reaches the model. Check it directly.

In [ ]:
probe = "you are an idiot"
print("clean     :", scorer.tokenizer.tokenize(probe))
print("zero_width:", scorer.tokenizer.tokenize(ATTACKS["zero_width"](probe)))
print("homoglyph :", scorer.tokenizer.tokenize(ATTACKS["homoglyph"](probe)))

Identical token sequences for `zero_width` confirm it. The defense is real but **incidental** — inherited from a 2018 preprocessing decision, not chosen. Nothing guarantees the next tokenizer keeps it.

`homoglyph` shows the opposite: the token sequence shatters, because Cyrillic `а е о` are distinct codepoints that fall out of vocabulary.

### Does a normalization pass recover it?

The cheapest possible defense: strip zero-width characters and reverse the homoglyph map before scoring. If that closes the gap, it belongs in preprocessing, not in the model.

In [ ]:
REVERSE = {v: k for k, v in HOMOGLYPHS.items()}

def normalize(text):
    return "".join(REVERSE.get(ch, ch) for ch in text.replace(ZWSP, ""))

for name in ["homoglyph", "zero_width"]:
    attacked = [ATTACKS[name](t) for t in toxic]
    raw = rate_above(scorer(attacked), shipping.threshold)
    fixed = rate_above(scorer([normalize(t) for t in attacked]), shipping.threshold)
    print(f"{name:>12}: recall {raw:.3f} -> {fixed:.3f} after normalization")

## 5. Findings

**1. Homoglyph substitution is the only attack that works — and it works completely.** Recall 0.780 → 0.303 at the shipping threshold; mean score 0.605 → 0.214. Cyrillic lookalikes are distinct codepoints, so tokens fall out of vocabulary and the model scores what is effectively a different language.

**2. The operating point decides whether the problem is visible.** The same attack on the same data reads as a 0.017 drop at the recall-pinned threshold and 0.477 at the FPR-pinned one — 28× difference in apparent severity. My first run measured only the former and concluded the model was robust. The tell was homoglyph showing a 0.017 recall drop while its mean score fell by two-thirds: a metric that cannot move is not evidence of safety.

**3. `zero_width` is not defended, it is deleted** — stripped by BERT's text cleaning before tokenization, as the token comparison above confirms. Incidental protection, not designed protection.

**4. `spaced` and `repeated` make the model *more* confident, which is a defect, not robustness.** Recall rises to 0.997 and 0.983, above clean. The model appears to read character fragmentation and vowel repetition as toxicity signals in themselves — plausibly learned from Jigsaw, where `s o   a n g r y` and `whyyyy` correlate with abuse. That predicts over-flagging of emphatic-but-benign text, a false-positive problem affecting real users. **Applying the attacks to the benign set to measure FPR shift is the most load-bearing gap in this result.**

**Limitations.** Single model, single dataset, English only. 300 examples per condition (±4pp, CIs reported). Attacks are hand-written rather than searched, so these are a lower bound. The homoglyph map covers 8 characters; a fuller one would likely be worse. `civil_comments` labels are crowd-sourced and carry annotator bias that propagates into the baseline.